In [16]:
from sts.stsig.feat import graph
from pathlib import Path
from sts.utils import configParser
from sts.mktdata.ticker import Ticker
import datetime as dt
import traceback
import pandas as pd

In [2]:
config_path = Path('/home/yuqing42/repos/systrading/sts/stsig/feat/config.yml')
config = configParser(config_path)

In [3]:
signal_graph = graph.SignalGraph(config)

In [4]:
ticker = Ticker('us/etf.yml')
symbols = ticker.top_trading_symbols(dt.date(2026, 4, 8))

In [21]:
output_folder = Path('/home/yuqing42/data/stsdata/research/mktstates/')
sdate = dt.date(2024, 1, 1)
edate = dt.date(2026, 4, 10)
failed_cases = []
result= []
n = 0
for symbol in symbols:
    n += 1
    try:
        signal_graph.run(symbol, sdate, edate, interval='1d')
        df = signal_graph.output()
        file_path = output_folder / f'{symbol}'
        res = df.iloc[-1].to_dict()
        res['symbol'] = symbol
        result.append(res)
        df.to_parquet(str(file_path))
    except Exception:
        traceback.print_exc()
        failed_cases.append(symbol)
    if n % 50 == 0:
        result1 = pd.DataFrame(result)
        result1.to_html('/home/yuqing42/Dropbox/stsinvest/mktstates.html')
        result1.to_csv('/home/yuqing42/Dropbox/stsinvest/mktstates.csv')

result1 = pd.DataFrame(result)
result1.to_html('/home/yuqing42/Dropbox/stsinvest/mktstates.html')
result1.to_csv('/home/yuqing42/Dropbox/stsinvest/mktstates.csv')

/home/yuqing42/data/stsdata/us/etfs/daily/20260408.parquet does not exist, skip
/home/yuqing42/data/stsdata/us/etfs/daily/20260409.parquet does not exist, skip
/home/yuqing42/data/stsdata/us/etfs/daily/20260410.parquet does not exist, skip
/home/yuqing42/data/stsdata/us/etfs/daily/20260408.parquet does not exist, skip
/home/yuqing42/data/stsdata/us/etfs/daily/20260409.parquet does not exist, skip
/home/yuqing42/data/stsdata/us/etfs/daily/20260410.parquet does not exist, skip
/home/yuqing42/data/stsdata/us/etfs/daily/20260408.parquet does not exist, skip
/home/yuqing42/data/stsdata/us/etfs/daily/20260409.parquet does not exist, skip
/home/yuqing42/data/stsdata/us/etfs/daily/20260410.parquet does not exist, skip
/home/yuqing42/data/stsdata/us/etfs/daily/20260408.parquet does not exist, skip
/home/yuqing42/data/stsdata/us/etfs/daily/20260409.parquet does not exist, skip
/home/yuqing42/data/stsdata/us/etfs/daily/20260410.parquet does not exist, skip
/home/yuqing42/data/stsdata/us/etfs/dail

In [ ]:
df = pd.read_parquet('')

In [25]:
me_df = ticker.history(symbols, dt.date(2026, 4, 8), dt.date(2026, 4, 8), interval = '1ME')[['symbol', 'volume', 'close']]

In [28]:
result1 = result1.merge(me_df, on = 'symbol')

In [33]:
result1['flag'] = result1['volume'] * result1['close'] > 100e6
result1 = result1.sort_values(['flag', 'trendvol10Len'])

In [34]:
result1.to_html('/home/yuqing42/Dropbox/stsinvest/mktstates.html')
result1.to_csv('/home/yuqing42/Dropbox/stsinvest/mktstates.csv')

In [32]:
result1

,bbcomp20,trendvol10,bbcomp20Len,trendvol10Len,symbol,volume,close,flag
0,0.484593,0.000000,238.0,0.0,AAA,21546.0,24.945000,False
1,0.083474,0.000000,0.0,0.0,AAAA,6821.0,26.919001,False
2,0.754339,0.303796,3.0,9.0,AAAC,213.0,19.985001,False
3,0.119629,0.000000,0.0,0.0,AAAU,7819645.0,46.090000,True
4,0.256902,0.424812,30.0,6.0,AACBR,10006.0,0.300000,False
...,...,...,...,...,...,...,...,...
6331,0.000000,0.000000,0.0,0.0,ZTOP,827.0,51.620399,False
6332,0.051341,0.000000,0.0,0.0,ZTR,337524.0,6.760000,False
6333,0.000000,0.000000,0.0,0.0,ZTRE,11222.0,50.770000,False
6334,0.420517,0.000000,3.0,0.0,ZTWO,5041.0,50.480000,False


In [ ]:
from sts.stsig.filter.vol import high_vol_filter, low_volume_filter
from sts.mktdata.ticker import Ticker
from sts.quant.candle import Candle
import datetime as dt
import pandas as pd
ticker = Ticker('us/etf.yml')
symbols = ticker.top_trading_symbols(dt.date(2026, 4, 8))

result = []
n = 0
for symbol in symbols:
    n += 1
    df = ticker.history(symbol, '20250101', '20260407')
    high_vol_score = high_vol_filter(df, window = 20)
    low_volume_score = low_volume_filter(df, window = 252, min_trade_value=100e6)
    result.append({'symbol':symbol, 'highVolScore':high_vol_score, 'lowVolumeScore': low_volume_score})
    if n%50==0:
        if len(result) > 0:
            result_df = pd.DataFrame(result)
            result_df.to_csv('/home/yuqing42/Dropbox/stsinvest/filterScore.csv')

if len(result) > 0:
    result_df = pd.DataFrame(result)
    result_df.to_csv('/home/yuqing42/Dropbox/stsinvest/filterScore.csv')
result = pd.DataFrame(result)
idx = (result['highVolScore'] > 0.5 ) | (result['lowVolumeScore'] > 0.5)
t = result[~idx]
t.to_parquet('~/data/stsdata/research/usetf/liquid_symbols')